# PlantDiseases — Training Notebook

Обучение двухступенчатого пайплайна обнаружения болезней растений.

| Стадия | Модель | Параметры | Задача |
|--------|--------|-----------|--------|
| **Stage 1 — Детектор** | MobileNetV3-Small | 1.5M | Бинарная: здоровое / больное + Grad-CAM регион |
| **Stage 2 — Классификатор** | EfficientNet-B0 | 4.0M | 9 классов болезней (PlantVillage Tomato) |

**Как использовать:**
1. Откройте в Google Colab
2. Включите GPU: `Runtime → Change runtime type → T4 GPU`
3. Запускайте ячейки последовательно сверху вниз
4. В конце скачайте готовые модели `detector.pth` и `classifier.pth`

## 1. Установка зависимостей и проверка GPU

In [ ]:
import torch, torchvision
print(f"PyTorch:      {torch.__version__}")
print(f"TorchVision:  {torchvision.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"Memory:       {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")

## 2. Загрузка датасета PlantVillage

Загружаем датасет с Kaggle прямо в Colab. Вам понадобится Kaggle API-токен (`kaggle.json`).

**Если у вас нет токена:**
1. Зайдите на https://www.kaggle.com/settings → Create New Token
2. Скачается файл `kaggle.json`
3. Загрузите его в Colab при запуске ячейки ниже

In [ ]:
import os, shutil, random
from pathlib import Path
from google.colab import files

# --- Загрузка kaggle.json если ещё нет ---
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

if not kaggle_json.exists():
    print("Загрузите файл kaggle.json:")
    uploaded = files.upload()
    for name, data in uploaded.items():
        kaggle_json.write_bytes(data)
    kaggle_json.chmod(0o600)
    print("kaggle.json сохранён!")

# --- Скачиваем датасет ---
!pip install -q kaggle
!kaggle datasets download -d emmarex/plantdisease -p /content/dataset --unzip

print("\nДатасет скачан!")
!ls /content/dataset/

## 3. Подготовка данных

PlantVillage содержит папки вида `Tomato___Early_blight`. Мы переименуем их в целевые классы из `CLASS_MAP` и разобьём на train/val (85/15). Число итоговых классов определяется тем, какие папки есть в датасете — для шипуемых весов это 9 классов (PlantVillage Tomato).

In [ ]:
import glob, shutil, random
from pathlib import Path

# ── Маппинг папок PlantVillage → наши классы ────────────────────────
# Ключ (lowercase подстрока в имени папки) → наш класс
CLASS_MAP = {
    "healthy":                          "healthy",
    "bacterial_spot":                   "bacterial_spot",
    "early_blight":                     "early_blight",
    "late_blight":                      "late_blight",
    "leaf_mold":                        "leaf_mold",
    "septoria_leaf_spot":               "septoria_leaf_spot",
    "spider_mites":                     "spider_mites",
    "target_spot":                      "target_spot",
    "tomato_mosaic_virus":              "mosaic_virus",
    "mosaic_virus":                     "mosaic_virus",
    "tomato_yellow_leaf_curl_virus":    "yellow_leaf_curl",
    "yellow_leaf_curl":                 "yellow_leaf_curl",
    "powdery_mildew":                   "powdery_mildew",
    "rust":                             "rust",
    "anthracnose":                      "anthracnose",
    "botrytis":                         "botrytis",
    "root_rot":                         "root_rot",
}

# Ищем скачанные папки
RAW_DIR = Path("/content/dataset")
# PlantVillage может быть в подпапке
candidates = list(RAW_DIR.rglob("*healthy*"))
if candidates:
    src_root = candidates[0].parent
else:
    for d in RAW_DIR.rglob("*"):
        if d.is_dir() and (list(d.glob("*.jpg")) or list(d.glob("*.JPG"))):
            src_root = d.parent
            break
    else:
        src_root = RAW_DIR

print(f"Исходные папки найдены в: {src_root}")
src_dirs = sorted([d for d in src_root.iterdir() if d.is_dir()])
print(f"Найдено {len(src_dirs)} папок:")
for d in src_dirs:
    n = len(list(d.glob("*.*")))
    print(f"  {d.name}: {n} файлов")

# ── Создаём data/train и data/val ───────────────────────────────────
DATA_DIR = Path("/content/data")
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

VAL_RATIO = 0.15
random.seed(42)
total_train, total_val = 0, 0
found_classes = set()

for src_dir in src_dirs:
    # Определяем наш класс — ищем совпадение подстроки
    folder_lower = src_dir.name.lower().replace(" ", "_").replace(",", "")
    target_class = None
    for pattern, cls_name in CLASS_MAP.items():
        if pattern in folder_lower:
            target_class = cls_name
            break

    if target_class is None:
        print(f"  [пропущено] {src_dir.name} — не соответствует ни одному классу")
        continue

    # Собираем все изображения
    images = []
    for ext in ("*.jpg", "*.JPG", "*.jpeg", "*.JPEG", "*.png", "*.PNG"):
        images.extend(src_dir.glob(ext))

    if not images:
        print(f"  [пропущено] {src_dir.name} — нет изображений")
        continue

    random.shuffle(images)
    split = int(len(images) * (1 - VAL_RATIO))
    train_imgs = images[:split]
    val_imgs = images[split:]

    train_cls_dir = TRAIN_DIR / target_class
    val_cls_dir = VAL_DIR / target_class
    train_cls_dir.mkdir(parents=True, exist_ok=True)
    val_cls_dir.mkdir(parents=True, exist_ok=True)

    for img in train_imgs:
        shutil.copy2(img, train_cls_dir / img.name)
    for img in val_imgs:
        shutil.copy2(img, val_cls_dir / img.name)

    total_train += len(train_imgs)
    total_val += len(val_imgs)
    found_classes.add(target_class)
    print(f"  {src_dir.name} → {target_class}: {len(train_imgs)} train / {len(val_imgs)} val")

# НЕ создаём пустых папок — ImageFolder требует чтобы в каждой папке были файлы

print(f"\nИтого: {total_train} train / {total_val} val")
print(f"Найдено классов с данными: {len(found_classes)}")
print(f"Классы: {sorted(found_classes)}")
print(f"\nГотово! Данные в {DATA_DIR}")

## 4. Конфигурация и общий код

Настройки обучения, аугментации, датасеты и функции тренировки — общие для обоих моделей.

In [ ]:
import copy, time, os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt

# ── Пути ─────────────────────────────────────────────────────────────
DATA_DIR   = Path("/content/data")
MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(exist_ok=True)

# ── Гиперпараметры ──────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2   # Colab обычно даёт 2 CPU

# ── Определяем классы динамически из данных ──────────────────────────
# ImageFolder использует отсортированные имена папок как классы
CLASS_NAMES = sorted([
    d.name for d in (DATA_DIR / "train").iterdir()
    if d.is_dir() and any(d.iterdir())  # только непустые папки
])
NUM_CLASSES = len(CLASS_NAMES)

print(f"Конфигурация:")
print(f"  IMG_SIZE    = {IMG_SIZE}")
print(f"  BATCH_SIZE  = {BATCH_SIZE}")
print(f"  NUM_CLASSES = {NUM_CLASSES}")
print(f"  Device      = {device}")
print(f"\nКлассы ({NUM_CLASSES}):")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i:>2}. {name}")

# ── Аугментации ─────────────────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
# ── Бинарный датасет (для детектора) ─────────────────────────────────

class BinaryPlantDataset(torch.utils.data.Dataset):
    """Оборачивает ImageFolder: healthy → 0, всё остальное → 1."""
    def __init__(self, image_folder, healthy_idx):
        self.dataset = image_folder
        self.healthy_idx = healthy_idx
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        return img, (0 if label == self.healthy_idx else 1)


def create_dataloaders(binary=False):
    """Создаёт train/val DataLoader. binary=True → 2 класса для детектора.

    Классы определяются автоматически из непустых подпапок.
    """
    train_ds = datasets.ImageFolder(DATA_DIR / "train", transform=train_transforms)
    val_ds   = datasets.ImageFolder(DATA_DIR / "val",   transform=val_transforms)

    print(f"  ImageFolder нашёл {len(train_ds.classes)} классов: {train_ds.classes}")

    if binary:
        h_idx = train_ds.class_to_idx.get("healthy", 0)
        train_ds = BinaryPlantDataset(train_ds, h_idx)
        val_raw  = datasets.ImageFolder(DATA_DIR / "val", transform=val_transforms)
        val_ds   = BinaryPlantDataset(val_raw, val_raw.class_to_idx.get("healthy", 0))
        names = ["healthy", "diseased"]
    else:
        names = list(train_ds.classes)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, names


# ── Функции обучения ─────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        loss = criterion(out, labels)
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


def train_model(model, train_loader, val_loader, *,
                epochs_freeze, epochs_finetune,
                lr_freeze, lr_finetune, unfreeze_from, name):
    """Двухфазное обучение: замороженный backbone → fine-tune верхних слоёв."""

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_acc, best_state, patience_cnt, patience = 0.0, None, 0, 7

    # ── Фаза 1: обучение головы ─────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  Фаза 1: обучение головы {name} ({epochs_freeze} эпох)")
    print(f"{'='*55}")

    opt = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=lr_freeze, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs_freeze)

    for ep in range(1, epochs_freeze + 1):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_loader, criterion, opt)
        vl, va = evaluate(model, val_loader, criterion)
        sched.step()
        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_acc"].append(ta);  history["val_acc"].append(va)
        print(f"  Epoch {ep:>2}/{epochs_freeze}  "
              f"train_loss={tl:.4f}  acc={ta:.3f}  "
              f"val_loss={vl:.4f}  acc={va:.3f}  ({time.time()-t0:.1f}s)")
        if va > best_acc:
            best_acc, best_state, patience_cnt = va, copy.deepcopy(model.state_dict()), 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"  Early stopping (epoch {ep})")
                break

    if best_state:
        model.load_state_dict(best_state)

    # ── Фаза 2: fine-tune ───────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  Фаза 2: fine-tune {name} ({epochs_finetune} эпох)")
    print(f"{'='*55}")

    params = list(model.parameters())
    for p in params[unfreeze_from:]:
        p.requires_grad = True

    opt = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=lr_finetune, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs_finetune)
    patience_cnt = 0

    for ep in range(1, epochs_finetune + 1):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_loader, criterion, opt)
        vl, va = evaluate(model, val_loader, criterion)
        sched.step()
        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_acc"].append(ta);  history["val_acc"].append(va)
        print(f"  Epoch {ep:>2}/{epochs_finetune}  "
              f"train_loss={tl:.4f}  acc={ta:.3f}  "
              f"val_loss={vl:.4f}  acc={va:.3f}  ({time.time()-t0:.1f}s)")
        if va > best_acc:
            best_acc, best_state, patience_cnt = va, copy.deepcopy(model.state_dict()), 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"  Early stopping (epoch {ep})")
                break

    if best_state:
        model.load_state_dict(best_state)

    print(f"\n  Лучшая val accuracy: {best_acc:.4f} ({best_acc*100:.1f}%)")
    return model, history

print("Функции обучения готовы.")

In [ ]:
# ── Визуализация и оценка ────────────────────────────────────────────

def plot_history(history, title=""):
    """Графики loss и accuracy за обе фазы обучения."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history["train_loss"], label="train")
    ax1.plot(history["val_loss"],   label="val")
    ax1.set_title(f"{title} — Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history["train_acc"], label="train")
    ax2.plot(history["val_acc"],   label="val")
    ax2.set_title(f"{title} — Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


@torch.no_grad()
def detailed_evaluation(model, loader, class_names):
    """Confusion matrix + per-class accuracy + top ошибок."""
    model.eval()
    n = len(class_names)
    confusion = np.zeros((n, n), dtype=int)

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = model(imgs).argmax(1)
        for t, p in zip(labels.cpu().numpy(), preds.cpu().numpy()):
            confusion[t][p] += 1

    total = confusion.sum()
    correct = confusion.trace()
    acc = correct / total if total > 0 else 0

    print(f"\nОбщая accuracy: {acc:.4f} ({acc*100:.1f}%)")
    print(f"\n{'Класс':<25} {'Accuracy':>10} {'Кол-во':>10}")
    print("-" * 50)
    for i, name in enumerate(class_names):
        row = confusion[i].sum()
        if row > 0:
            print(f"{name:<25} {confusion[i][i]/row:>9.1%} {row:>10}")
        else:
            print(f"{name:<25} {'N/A':>10} {0:>10}")

    # Confusion matrix heatmap
    fig, ax = plt.subplots(figsize=(max(8, n * 0.8), max(6, n * 0.6)))
    im = ax.imshow(confusion, cmap="Blues")
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(class_names, fontsize=8)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix (accuracy={acc:.1%})")
    plt.colorbar(im, ax=ax)

    # Числа в ячейках
    for i in range(n):
        for j in range(n):
            if confusion[i][j] > 0:
                ax.text(j, i, str(confusion[i][j]), ha="center", va="center",
                        fontsize=7, color="white" if confusion[i][j] > confusion.max()/2 else "black")
    plt.tight_layout()
    plt.show()

    # Top ошибок
    errors = []
    for i in range(n):
        for j in range(n):
            if i != j and confusion[i][j] > 0:
                errors.append((confusion[i][j], class_names[i], class_names[j]))
    errors.sort(reverse=True)
    if errors:
        print("\nТоп ошибок:")
        for cnt, true_n, pred_n in errors[:10]:
            print(f"  {true_n} → {pred_n}: {cnt}")

    return acc

print("Функции визуализации готовы.")

---
## 5. Stage 1 — Обучение детектора (MobileNetV3-Small)

Бинарный классификатор: **здоровое** vs **больное** растение.
Быстрый и лёгкий (~1.5M параметров с бинарной головой) — при инференсе используется с Grad-CAM для локализации поражённого участка.

In [ ]:
# ── Данные для детектора (бинарные) ──────────────────────────────────
det_train, det_val, det_names = create_dataloaders(binary=True)

print(f"Детектор — данные:")
print(f"  Train: {len(det_train.dataset)} изображений")
print(f"  Val:   {len(det_val.dataset)} изображений")
print(f"  Классы: {det_names}")

In [ ]:
# ── Строим и обучаем детектор ─────────────────────────────────────────
detector = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

# Замораживаем backbone
for p in detector.features.parameters():
    p.requires_grad = False

# Заменяем голову на 2 класса
detector.classifier[-1] = nn.Linear(detector.classifier[-1].in_features, 2)
detector = detector.to(device)

total_p = sum(p.numel() for p in detector.parameters())
train_p = sum(p.numel() for p in detector.parameters() if p.requires_grad)
print(f"MobileNetV3-Small: {total_p:,} параметров ({train_p:,} обучаемых)")

detector, det_history = train_model(
    detector, det_train, det_val,
    epochs_freeze=15,
    epochs_finetune=10,
    lr_freeze=1e-3,
    lr_finetune=1e-5,
    unfreeze_from=-40,
    name="Детектор",
)

In [ ]:
# ── Оценка и графики детектора ────────────────────────────────────────
plot_history(det_history, "Stage 1 — Детектор")
det_acc = detailed_evaluation(detector, det_val, det_names)

# Сохраняем
DET_PATH = MODELS_DIR / "detector.pth"
torch.save(detector.state_dict(), str(DET_PATH))
print(f"\nДетектор сохранён → {DET_PATH}  (accuracy: {det_acc:.1%})")

---
## 6. Stage 2 — Обучение классификатора (EfficientNet-B0)

Fine-grained классификатор болезней по числу найденных в данных классов (в шипуемой версии — 9). Работает на обрезанном ROI из Stage 1, что повышает точность за счёт фокуса на поражённом участке.

In [ ]:
# ── Данные для классификатора (число классов = сколько непустых папок нашёл ImageFolder) ──
cls_train, cls_val, cls_names = create_dataloaders(binary=False)

print(f"Классификатор — данные:")
print(f"  Train: {len(cls_train.dataset)} изображений")
print(f"  Val:   {len(cls_val.dataset)} изображений")
print(f"  Классы ({len(cls_names)}):")
for i, name in enumerate(cls_names):
    print(f"    {i:>2}. {name}")

In [ ]:
# ── Строим и обучаем классификатор ───────────────────────────────────
classifier = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Замораживаем backbone
for p in classifier.features.parameters():
    p.requires_grad = False

# Заменяем голову — количество классов берём из данных
n_classes = len(cls_names)
classifier.classifier[-1] = nn.Linear(classifier.classifier[-1].in_features, n_classes)
classifier = classifier.to(device)

total_p = sum(p.numel() for p in classifier.parameters())
train_p = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
print(f"EfficientNet-B0: {total_p:,} параметров ({train_p:,} обучаемых)")
print(f"Количество классов: {n_classes}")

classifier, cls_history = train_model(
    classifier, cls_train, cls_val,
    epochs_freeze=20,
    epochs_finetune=15,
    lr_freeze=1e-3,
    lr_finetune=5e-6,
    unfreeze_from=-60,
    name="Классификатор",
)

In [ ]:
# ── Оценка и графики классификатора ──────────────────────────────────
plot_history(cls_history, "Stage 2 — Классификатор")
cls_acc = detailed_evaluation(classifier, cls_val, cls_names)

# Сохраняем
CLS_PATH = MODELS_DIR / "classifier.pth"
torch.save(classifier.state_dict(), str(CLS_PATH))
print(f"\nКлассификатор сохранён → {CLS_PATH}  (accuracy: {cls_acc:.1%})")

---
## 7. Тест пайплайна на примере

Проверим обе модели вместе: детектор находит участок, классификатор определяет болезнь.

In [ ]:
import torch.nn.functional as F
from PIL import Image
import cv2

# ── Grad-CAM для визуализации ────────────────────────────────────────
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    @torch.enable_grad()
    def generate(self, input_tensor, target_class=1):
        self.model.zero_grad()
        output = self.model(input_tensor)
        output[0, target_class].backward()
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam

# ── Берём случайную картинку из val ──────────────────────────────────
val_dir = DATA_DIR / "val"
all_imgs = []
for cls_dir in val_dir.iterdir():
    if cls_dir.is_dir():
        for f in cls_dir.glob("*.*"):
            all_imgs.append((f, cls_dir.name))

random.shuffle(all_imgs)
test_samples = all_imgs[:6]

inference_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

grad_cam = GradCAM(detector, detector.features[-1])

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, (img_path, true_cls) in enumerate(test_samples):
    ax = axes[idx // 3][idx % 3]
    img = Image.open(img_path).convert("RGB")
    tensor = inference_tf(img).unsqueeze(0).to(device)

    # Stage 1: детектор
    with torch.no_grad():
        det_logits = detector(tensor)
    det_probs = F.softmax(det_logits, dim=1)[0]
    is_diseased = det_probs[1] > det_probs[0]

    if is_diseased:
        # Grad-CAM heatmap
        heatmap = grad_cam.generate(tensor, target_class=1)
        heatmap_resized = cv2.resize(heatmap, (img.width, img.height))

        # Stage 2: классификатор
        with torch.no_grad():
            cls_logits = classifier(tensor)
        cls_probs = F.softmax(cls_logits, dim=1)[0]
        pred_idx = cls_probs.argmax().item()
        pred_name = cls_names[pred_idx]
        pred_conf = cls_probs[pred_idx].item()

        # Визуализация с heatmap overlay
        img_np = np.array(img)
        heatmap_color = cv2.applyColorMap(
            (heatmap_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
        heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
        overlay = (img_np * 0.6 + heatmap_color * 0.4).astype(np.uint8)

        ax.imshow(overlay)
        ax.set_title(f"Pred: {pred_name} ({pred_conf:.0%})\nTrue: {true_cls}",
                     fontsize=9, color="red" if pred_name != true_cls else "green")
    else:
        ax.imshow(img)
        ax.set_title(f"Pred: healthy ({det_probs[0]:.0%})\nTrue: {true_cls}",
                     fontsize=9, color="green" if true_cls == "healthy" else "red")

    ax.axis("off")

plt.suptitle("Тест пайплайна: детектор + Grad-CAM + классификатор", fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Скачать обученные модели

Скачайте `detector.pth` и `classifier.pth` и положите их в папку `server/models/` вашего проекта.
Сервер подхватит их автоматически при запуске.

In [ ]:
import json
from google.colab import files

# Сохраняем маппинг классов рядом с моделями (сервер должен знать порядок)
meta = {"class_names": cls_names, "num_classes": len(cls_names)}
meta_path = MODELS_DIR / "classes.json"
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2))

det_size = os.path.getsize(DET_PATH) / 1e6
cls_size = os.path.getsize(CLS_PATH) / 1e6

print(f"Готовые файлы:")
print(f"  detector.pth   — {det_size:.1f} MB  (accuracy: {det_acc:.1%})")
print(f"  classifier.pth — {cls_size:.1f} MB  (accuracy: {cls_acc:.1%})")
print(f"  classes.json   — маппинг {len(cls_names)} классов")
print(f"  Суммарно:        {det_size + cls_size:.1f} MB")
print()
print("Скачивание...")

files.download(str(DET_PATH))
files.download(str(CLS_PATH))
files.download(str(meta_path))

print()
print("Положите все 3 файла в server/models/ и запустите сервер:")
print("  Linux:   ./start.sh")
print("  Windows: start.bat")

---
## (Опционально) Сохранить на Google Drive

Если не хотите скачивать через браузер — можно сохранить прямо на Google Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

save_dir = Path("/content/drive/MyDrive/PlantDiseases_Models")
save_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(DET_PATH, save_dir / "detector.pth")
shutil.copy2(CLS_PATH, save_dir / "classifier.pth")
shutil.copy2(meta_path, save_dir / "classes.json")

print(f"Модели сохранены в Google Drive:")
print(f"  {save_dir / 'detector.pth'}")
print(f"  {save_dir / 'classifier.pth'}")
print(f"  {save_dir / 'classes.json'}")